In [104]:
# Cai package can thiet neu thieu (chay 1 lan dau notebook)
import sys
import subprocess
import importlib.util

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "tensorflow": "tensorflow",
    "cv2": "opencv-python",
}

for module_name, pip_name in required.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

print("Dependency check complete.")

Dependency check complete.


In [105]:
import os
import random
import time
import warnings
import numpy as np

# Dat bien moi truong truoc khi import tensorflow
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Ep TensorFlow chi dung CPU de tranh loi/warning CUDA
try:
    tf.config.set_visible_devices([], "GPU")
except Exception:
    pass

# Duong dan du lieu tren Kaggle
DATA_ROOT = "C:/Users/Admin/Desktop/Edge AI/Edge-AI-challenge/edge-ai-challenge-2026-warm-up/kaggle_testing"
TRAIN_PATH = os.path.join(DATA_ROOT, "train")
TEST_DIR = os.path.join(DATA_ROOT, "test")

IMG_SIZE = (32, 32)
BATCH_SIZE = 128
NUM_CLASSES = 10
AUTOTUNE = tf.data.AUTOTUNE

print("TensorFlow:", tf.__version__)
print("TRAIN_PATH exists:", os.path.exists(TRAIN_PATH))
print("TEST_DIR exists:", os.path.exists(TEST_DIR))
print("GPU devices:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.21.0
TRAIN_PATH exists: True
TEST_DIR exists: True
GPU devices: []


In [106]:
# Tao train/validation split truc tiep tu thu muc train
train_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="rgb",
    label_mode="int",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="rgb",
    label_mode="int",
)

class_names = train_raw.class_names
print("Classes:", class_names)
print("So class:", len(class_names))

Found 2000 files belonging to 10 classes.
Using 1600 files for training.
Found 2000 files belonging to 10 classes.
Using 400 files for validation.
Classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
So class: 10


In [107]:
# Pipeline du lieu toi uu cho huan luyen local
train_ds = train_raw.shuffle(4096, seed=SEED, reshuffle_each_iteration=True).cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

for x_batch, y_batch in train_ds.take(1):
    print("Train batch shape:", x_batch.shape, y_batch.shape, x_batch.dtype)
    print("Label sample:", y_batch[:10].numpy())

Train batch shape: (128, 32, 32, 3) (128,) <dtype: 'float32'>
Label sample: [1 2 1 1 7 6 8 5 5 9]


In [108]:
from tensorflow.keras import layers, models

def sep_block(x, filters, pool=True, drop=0.0, name="blk"):
    x = layers.SeparableConv2D(filters, 3, padding="same", use_bias=True, name=f"{name}_sep")(x)
    x = layers.LayerNormalization(name=f"{name}_ln")(x)
    x = layers.Activation("relu", name=f"{name}_relu")(x)
    if pool:
        x = layers.MaxPooling2D(2, name=f"{name}_pool")(x)
    if drop > 0:
        x = layers.Dropout(drop, seed=SEED, name=f"{name}_drop")(x)
    return x

def build_tiny_traffic_cnn(input_shape=(32, 32, 3), num_classes=10):
    inputs = layers.Input(shape=input_shape, name="input_rgb888")
    x = layers.Rescaling(1.0 / 255.0, name="rescale")(inputs)

    # Accuracy-first, van giu constraint <200k params
    x = layers.Conv2D(52, 3, padding="same", activation="relu", name="stem_conv")(x)
    x = sep_block(x, 72, pool=True, drop=0.03, name="b1")    # 16x16
    x = sep_block(x, 104, pool=True, drop=0.05, name="b2")   # 8x8
    x = sep_block(x, 136, pool=True, drop=0.06, name="b3")   # 4x4
    x = sep_block(x, 160, pool=False, drop=0.08, name="b4")
    x = sep_block(x, 192, pool=False, drop=0.08, name="b5")

    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(256, activation="relu", name="fc1")(x)
    x = layers.Dropout(0.12, seed=SEED, name="head_drop1")(x)
    x = layers.Dense(128, activation="relu", name="fc2")(x)
    x = layers.Dropout(0.10, seed=SEED, name="head_drop2")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="pred")(x)

    return models.Model(inputs, outputs, name="tiny_traffic_cnn")

model = build_tiny_traffic_cnn(input_shape=(32, 32, 3), num_classes=NUM_CLASSES)
model.summary()

param_count = model.count_params()
print("Total params:", param_count)
assert param_count < 200_000, f"Model vuot gioi han tham so: {param_count}"

Model: "tiny_traffic_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_rgb888 (InputLayer)       │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescale (Rescaling)             │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ stem_conv (Conv2D)              │ (None, 32, 32, 52)     │         1,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b1_sep (SeparableConv2D)        │ (None, 32, 32, 72)     │         4,284 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b1_ln (LayerNormalization)      │ (None, 32, 32, 72)     │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b1_relu (Activation)            │ (None, 32, 32, 72)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b1_pool (MaxPooling2D)          │ (None, 16, 16, 72)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b1_drop (Dropout)               │ (None, 16, 16, 72)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b2_sep (SeparableConv2D)        │ (None, 16, 16, 104)    │         8,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b2_ln (LayerNormalization)      │ (None, 16, 16, 104)    │           208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b2_relu (Activation)            │ (None, 16, 16, 104)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b2_pool (MaxPooling2D)          │ (None, 8, 8, 104)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b2_drop (Dropout)               │ (None, 8, 8, 104)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b3_sep (SeparableConv2D)        │ (None, 8, 8, 136)      │        15,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b3_ln (LayerNormalization)      │ (None, 8, 8, 136)      │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b3_relu (Activation)            │ (None, 8, 8, 136)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b3_pool (MaxPooling2D)          │ (None, 4, 4, 136)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b3_drop (Dropout)               │ (None, 4, 4, 136)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b4_sep (SeparableConv2D)        │ (None, 4, 4, 160)      │        23,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b4_ln (LayerNormalization)      │ (None, 4, 4, 160)      │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b4_relu (Activation)            │ (None, 4, 4, 160)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b4_drop (Dropout)               │ (None, 4, 4, 160)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b5_sep (SeparableConv2D)        │ (None, 4, 4, 192)      │        32,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b5_ln (LayerNormalization)      │ (None, 4, 4, 192)      │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b5_relu (Activation)            │ (None, 4, 4, 192)      │             

 Total params: 169,614 (662.55 KB)

 Trainable params: 169,614 (662.55 KB)

 Non-trainable params: 0 (0.00 B)

Total params: 169614


In [109]:
from tensorflow.keras import callbacks

# Phase 1: hoc nhanh
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

cb1 = [
    callbacks.ReduceLROnPlateau(
        monitor="val_accuracy", factor=0.6, patience=3, verbose=1, min_lr=5e-5
    ),
    callbacks.EarlyStopping(
        monitor="val_accuracy", patience=8, restore_best_weights=True, verbose=1
    ),
    callbacks.ModelCheckpoint(
        "best_tiny_traffic.keras", monitor="val_accuracy", save_best_only=True, verbose=1
    ),
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=cb1,
    verbose=1,
)

# Phase 2: fine-tune nhe de vut len mốc accuracy cuoi
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

cb2 = [
    callbacks.ReduceLROnPlateau(
        monitor="val_accuracy", factor=0.6, patience=3, verbose=1, min_lr=1e-5
    ),
    callbacks.EarlyStopping(
        monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1
    ),
    callbacks.ModelCheckpoint(
        "best_tiny_traffic.keras", monitor="val_accuracy", save_best_only=True, verbose=1
    ),
]

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=35,
    callbacks=cb2,
    verbose=1,
)

train_loss, train_acc = model.evaluate(train_raw, verbose=0)
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f"Train accuracy: {train_acc:.4f}")
print(f"Validation accuracy: {val_acc:.4f}")

Epoch 1/50


13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - accuracy: 0.1026 - loss: 2.3742
Epoch 1: val_accuracy improved from None to 0.10250, saving model to best_tiny_traffic.keras

Epoch 1: finished saving model to best_tiny_traffic.keras
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 243ms/step - accuracy: 0.0931 - loss: 2.3437 - val_accuracy: 0.1025 - val_loss: 2.3006 - learning_rate: 0.0010
Epoch 2/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step - accuracy: 0.1258 - loss: 2.2864
Epoch 2: val_accuracy improved from 0.10250 to 0.23500, saving model to best_tiny_traffic.keras

Epoch 2: finished saving model to best_tiny_traffic.keras
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 242ms/step - accuracy: 0.1388 - loss: 2.2570 - val_accuracy: 0.2350 - val_loss: 2.0912 - learning_rate: 0.0010
Epoch 3/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step - accuracy: 0.2176 - loss: 2.0415
Epoch 3: val_accuracy did not improve from 0.23500
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 219ms/step - accuracy: 0.2169 - loss: 1.9875 - val_accuracy: 0.2250 - val_loss: 1.8

In [110]:
# Luu model goc
model.save("tiny_traffic_cnn.h5")
print("Saved tiny_traffic_cnn.h5")

# Dai dien du lieu cho quantization
def representative_data_gen():
    for images, _ in train_raw.take(200):
        images = tf.cast(images, tf.float32)
        for i in range(images.shape[0]):
            yield [tf.expand_dims(images[i], axis=0)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8   # phu hop RGB888
converter.inference_output_type = tf.uint8

tflite_int8 = converter.convert()
with open("tiny_traffic_cnn_int8.tflite", "wb") as f:
    f.write(tflite_int8)

print("Saved tiny_traffic_cnn_int8.tflite")
print("TFLite size (KB):", round(len(tflite_int8) / 1024, 2))

Saved tiny_traffic_cnn.h5
INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmp0872c0ff\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmp0872c0ff\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmp0872c0ff'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 3), dtype=tf.float32, name='input_rgb888')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  2573415746064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2573415745488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2573415745872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2573415747600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2573415745680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2573415748368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2573415747408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2573415748560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2573415747024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2573415747216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  257341

In [111]:
import cv2
import pandas as pd

# Sap xep theo ten so de khop Id submit
test_files = sorted(os.listdir(TEST_DIR), key=lambda x: int(os.path.splitext(x)[0]))

X_test = []
for fname in test_files:
    img_path = os.path.join(TEST_DIR, fname)
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img is None:
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA)
    X_test.append(img)

X_test = np.asarray(X_test, dtype=np.uint8)
print("X_test:", X_test.shape, X_test.dtype)

X_test: (1000, 32, 32, 3) uint8


In [112]:
# Du doan bang model TFLite INT8/UINT8
interpreter = tf.lite.Interpreter(model_path="tiny_traffic_cnn_int8.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]
print("Input dtype:", input_details["dtype"], "shape:", input_details["shape"])
print("Output dtype:", output_details["dtype"], "shape:", output_details["shape"])

preds = []
for i in range(len(X_test)):
    x = X_test[i:i+1]

    if input_details["dtype"] == np.uint8:
        x_in = x.astype(np.uint8)
    else:
        x_in = x.astype(np.float32)

    interpreter.set_tensor(input_details["index"], x_in)
    interpreter.invoke()
    out = interpreter.get_tensor(output_details["index"])[0]
    preds.append(int(np.argmax(out)))

actual_ids = [f"{int(os.path.splitext(f)[0]):05d}" for f in test_files]
submission = pd.DataFrame({"Id": actual_ids, "Label": preds})
submission.to_csv("submission.csv", index=False)

print("Saved submission.csv")
print(submission.head())

Input dtype: <class 'numpy.uint8'> shape: [ 1 32 32  3]
Output dtype: <class 'numpy.uint8'> shape: [ 1 10]
Saved submission.csv
      Id  Label
0  00000      4
1  00007      2
2  00012      2
3  00014      6
4  00022      1


In [113]:
# Benchmark toc do suy luan tren host (chi de tham khao)
interpreter = tf.lite.Interpreter(model_path="tiny_traffic_cnn_int8.tflite")
interpreter.allocate_tensors()
in_det = interpreter.get_input_details()[0]
out_det = interpreter.get_output_details()[0]

sample_n = min(300, len(X_test))
sample = X_test[:sample_n]

# Warmup
for i in range(min(20, sample_n)):
    x = sample[i:i+1].astype(in_det["dtype"])
    interpreter.set_tensor(in_det["index"], x)
    interpreter.invoke()
    _ = interpreter.get_tensor(out_det["index"])

start = time.perf_counter()
for i in range(sample_n):
    x = sample[i:i+1].astype(in_det["dtype"])
    interpreter.set_tensor(in_det["index"], x)
    interpreter.invoke()
    _ = interpreter.get_tensor(out_det["index"])

elapsed = time.perf_counter() - start
ms_per_img = (elapsed / sample_n) * 1000

datasheet = {
    "model_name": "tiny_traffic_cnn",
    "input": "RGB888 32x32",
    "classes": NUM_CLASSES,
    "params": int(model.count_params()),
    "constraint_lt_200k": bool(model.count_params() < 200_000),
    "val_accuracy": float(val_acc),
    "tflite_file": "tiny_traffic_cnn_int8.tflite",
    "tflite_size_kb": round(os.path.getsize("tiny_traffic_cnn_int8.tflite") / 1024, 2),
    "host_inference_ms_per_image": round(ms_per_img, 3),
}

print(datasheet)

{'model_name': 'tiny_traffic_cnn', 'input': 'RGB888 32x32', 'classes': 10, 'params': 169614, 'constraint_lt_200k': True, 'val_accuracy': 1.0, 'tflite_file': 'tiny_traffic_cnn_int8.tflite', 'tflite_size_kb': 234.26, 'host_inference_ms_per_image': 1.299}


In [114]:
print("Pipeline complete.")
print("Artifacts:")
print("- tiny_traffic_cnn.h5")
print("- tiny_traffic_cnn_int8.tflite")
print("- submission.csv")

Pipeline complete.
Artifacts:
- tiny_traffic_cnn.h5
- tiny_traffic_cnn_int8.tflite
- submission.csv


In [115]:
# Diagnostic: kiem tra vi sao accuracy thap
import numpy as np
import tensorflow as tf

print("val_acc variable:", float(val_acc))
print("num classes from train_raw:", len(class_names), class_names)

# Dem phan bo nhan train/val
train_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, y in train_raw:
    y_np = y.numpy()
    train_counts += np.bincount(y_np, minlength=NUM_CLASSES)

val_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, y in val_ds:
    y_np = y.numpy()
    val_counts += np.bincount(y_np, minlength=NUM_CLASSES)

print("train_counts:", train_counts.tolist())
print("val_counts:", val_counts.tolist())

# Kiem tra baseline doan 1 class pho bien nhat
majority_cls = int(np.argmax(train_counts))
majority_acc = float(val_counts[majority_cls] / np.sum(val_counts))
print("majority class:", majority_cls, "baseline val acc:", round(majority_acc, 4))

# Danh gia model tren val 1 lan nua
eval_loss, eval_acc = model.evaluate(val_ds, verbose=0)
print("model.evaluate val_acc:", round(float(eval_acc), 4))

# Kiem tra output cua model tren 1 batch
x1, y1 = next(iter(val_ds))
p1 = model.predict(x1, verbose=0)
pred1 = np.argmax(p1, axis=1)
print("sample batch acc:", round(float(np.mean(pred1 == y1.numpy())), 4))

val_acc variable: 1.0
num classes from train_raw: 10 ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
train_counts: [166, 163, 165, 158, 166, 157, 162, 153, 152, 158]
val_counts: [34, 37, 35, 42, 34, 43, 38, 47, 48, 42]
majority class: 0 baseline val acc: 0.085
model.evaluate val_acc: 1.0
sample batch acc: 1.0


In [116]:
# Diagnostic tiep: xem lich su train
print("history keys:", list(history.history.keys()))
print("train acc first 5:", [round(float(x),4) for x in history.history.get("accuracy", [])[:5]])
print("train acc last 5:", [round(float(x),4) for x in history.history.get("accuracy", [])[-5:]])
print("val acc first 5:", [round(float(x),4) for x in history.history.get("val_accuracy", [])[:5]])
print("val acc last 5:", [round(float(x),4) for x in history.history.get("val_accuracy", [])[-5:]])
print("epochs trained:", len(history.history.get("accuracy", [])))

history keys: ['accuracy', 'loss', 'val_accuracy', 'val_loss', 'learning_rate']
train acc first 5: [0.1044, 0.1762, 0.2006, 0.2325, 0.235]
train acc last 5: [0.9869, 0.9881, 0.9844, 0.98, 0.9862]
val acc first 5: [0.13, 0.23, 0.245, 0.3625, 0.305]
val acc last 5: [0.965, 0.9675, 0.9675, 0.9675, 0.97]
epochs trained: 75


In [117]:
# Diagnostic tiep: phan bo du doan tren validation
all_pred = []
all_true = []
for xb, yb in val_ds:
    pb = model.predict(xb, verbose=0)
    all_pred.append(np.argmax(pb, axis=1))
    all_true.append(yb.numpy())

all_pred = np.concatenate(all_pred)
all_true = np.concatenate(all_true)

pred_counts = np.bincount(all_pred, minlength=NUM_CLASSES)
true_counts = np.bincount(all_true, minlength=NUM_CLASSES)
print("pred_counts:", pred_counts.tolist())
print("true_counts:", true_counts.tolist())
print("val acc recompute:", round(float(np.mean(all_pred == all_true)), 4))

pred_counts: [34, 37, 35, 42, 34, 43, 38, 47, 48, 42]
true_counts: [34, 37, 35, 42, 34, 43, 38, 47, 48, 42]
val acc recompute: 1.0


In [118]:
# Extra fine-tune de thu chot moc 1.0000 tren val
from tensorflow.keras import callbacks

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=8e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

ft_cb = [
    callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.5, patience=2, verbose=1, min_lr=1e-6),
    callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint("best_tiny_traffic.keras", monitor="val_accuracy", save_best_only=True, verbose=1),
]

_ = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=ft_cb, verbose=1)
val_acc = model.evaluate(val_ds, verbose=0)[1]
print("Post-finetune val_acc:", round(float(val_acc), 4))

Epoch 1/20


13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - accuracy: 1.0000 - loss: 0.0027
Epoch 1: val_accuracy improved from None to 1.00000, saving model to best_tiny_traffic.keras

Epoch 1: finished saving model to best_tiny_traffic.keras
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 243ms/step - accuracy: 1.0000 - loss: 0.0032 - val_accuracy: 1.0000 - val_loss: 0.0015 - learning_rate: 8.0000e-05
Epoch 2/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step - accuracy: 0.9997 - loss: 0.0027
Epoch 2: val_accuracy did not improve from 1.00000
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 209ms/step - accuracy: 0.9994 - loss: 0.0038 - val_accuracy: 1.0000 - val_loss: 0.0010 - learning_rate: 8.0000e-05
Epoch 3/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step - accuracy: 1.0000 - loss: 0.0035
Epoch 3: ReduceLROnPlateau reducing learning rate to 3.9999998989515007e-05.

Epoch 3: val_accuracy did not improve from 1.00000
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 217ms/step - accuracy: 1.0000 - loss: 0.0041 - val_accuracy: 1.0000 - val_loss: 0.0035 - learning_rat

In [119]:
# Diagnostic tiep: do chinh xac tren tap train khong augment
train_eval_loss, train_eval_acc = model.evaluate(train_raw, verbose=0)
print("model.evaluate train_raw acc:", round(float(train_eval_acc), 4))

model.evaluate train_raw acc: 1.0


In [120]:
# Quick sanity retrain: bo map augmentation ben ngoai, train pipeline co ban
train_plain = train_raw.shuffle(2048, seed=SEED).prefetch(AUTOTUNE)
val_plain = val_ds.prefetch(AUTOTUNE)

test_model = build_tiny_traffic_cnn(input_shape=(32, 32, 3), num_classes=NUM_CLASSES)
test_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

test_hist = test_model.fit(
    train_plain,
    validation_data=val_plain,
    epochs=8,
    verbose=1,
)

t_loss, t_acc = test_model.evaluate(train_raw, verbose=0)
v_loss, v_acc = test_model.evaluate(val_plain, verbose=0)
print("sanity train_raw acc:", round(float(t_acc), 4))
print("sanity val acc:", round(float(v_acc), 4))

Epoch 1/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 240ms/step - accuracy: 0.1006 - loss: 2.3140 - val_accuracy: 0.1450 - val_loss: 2.2540
Epoch 2/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 212ms/step - accuracy: 0.2000 - loss: 2.2090 - val_accuracy: 0.2650 - val_loss: 2.0631
Epoch 3/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 212ms/step - accuracy: 0.2387 - loss: 2.0104 - val_accuracy: 0.2675 - val_loss: 1.8469
Epoch 4/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 218ms/step - accuracy: 0.2713 - loss: 1.8179 - val_accuracy: 0.3450 - val_loss: 1.6847
Epoch 5/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 214ms/step - accuracy: 0.3100 - loss: 1.6599 - val_accuracy: 0.3775 - val_loss: 1.5687
Epoch 6/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 233ms/step - accuracy: 0.3344 - loss: 1.5924 - val_accuracy: 0.4100 - val_loss: 1.5257
Epoch 7/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 208ms/step - accuracy: 0.4000 - loss: 1.4917 - val_accuracy: 0.4700 - val_loss: 1.4212
Epoch 8/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 213ms/step - accuracy: 0.4319 - loss: 1.4154 - val_accuracy: 0.5675 - v

In [121]:
# Diagnostic tiep: kiem tra nhan trong dataset
import numpy as np
all_y = []
for _, yb in train_raw.take(20):
    all_y.append(yb.numpy())
all_y = np.concatenate(all_y)
print("train label min/max:", int(all_y.min()), int(all_y.max()))
print("train unique:", sorted(np.unique(all_y).tolist()))

train label min/max: 0 9
train unique: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [122]:
# Quick sanity retrain 2: model khong BatchNorm de tranh collapse infer
from tensorflow.keras import layers, models

def build_tiny_no_bn(input_shape=(32,32,3), num_classes=10):
    inp = layers.Input(shape=input_shape)
    x = layers.Rescaling(1./255.0)(inp)

    x = layers.Conv2D(24, 3, padding="same", activation="relu")(x)
    x = layers.DepthwiseConv2D(3, strides=2, padding="same", activation="relu")(x)
    x = layers.Conv2D(32, 1, padding="same", activation="relu")(x)

    x = layers.DepthwiseConv2D(3, strides=2, padding="same", activation="relu")(x)
    x = layers.Conv2D(48, 1, padding="same", activation="relu")(x)

    x = layers.DepthwiseConv2D(3, strides=2, padding="same", activation="relu")(x)
    x = layers.Conv2D(64, 1, padding="same", activation="relu")(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inp, out)

model2 = build_tiny_no_bn(num_classes=NUM_CLASSES)
print("model2 params:", model2.count_params())
model2.compile(optimizer=tf.keras.optimizers.Adam(3e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"] )

h2 = model2.fit(train_raw.shuffle(2048, seed=SEED), validation_data=val_ds, epochs=12, verbose=0)
tr2 = model2.evaluate(train_raw, verbose=0)[1]
va2 = model2.evaluate(val_ds, verbose=0)[1]
print("model2 train acc:", round(float(tr2),4), "val acc:", round(float(va2),4))
print("model2 val acc curve:", [round(float(v),4) for v in h2.history["val_accuracy"]])

model2 params: 12042
model2 train acc: 0.1037 val acc: 0.085
model2 val acc curve: [0.105, 0.085, 0.085, 0.085, 0.085, 0.085, 0.085, 0.085, 0.085, 0.085, 0.085, 0.085]


In [123]:
# Auto search nhanh mot vai cau hinh de tranh mode collapse
import tensorflow as tf
from tensorflow.keras import layers, models

def make_model(cfg):
    inp = layers.Input(shape=(32,32,3))
    x = layers.Rescaling(1./255.0)(inp)
    for i, c in enumerate(cfg["conv"]):
        x = layers.Conv2D(c, 3, padding="same", activation="relu")(x)
        if i in cfg.get("pool_at", []):
            x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(cfg.get("drop", 0.2))(x)
    x = layers.Dense(cfg.get("fc", 64), activation="relu")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    m = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(cfg.get("lr", 3e-4)),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

candidates = [
    {"name":"A","conv":[24,32,48,64], "pool_at":[0,1,2], "fc":64, "drop":0.2, "lr":3e-4},
    {"name":"B","conv":[32,48,64,96], "pool_at":[0,1,2], "fc":96, "drop":0.2, "lr":3e-4},
    {"name":"C","conv":[32,32,64,64], "pool_at":[0,1,2], "fc":64, "drop":0.1, "lr":1e-3},
]

results = []
for cfg in candidates:
    m = make_model(cfg)
    m.fit(train_raw.shuffle(2048, seed=SEED), validation_data=val_ds, epochs=12, verbose=0)
    tr = m.evaluate(train_raw, verbose=0)[1]
    va = m.evaluate(val_ds, verbose=0)[1]
    pred_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
    for xb, _ in val_ds:
        pred = np.argmax(m.predict(xb, verbose=0), axis=1)
        pred_counts += np.bincount(pred, minlength=NUM_CLASSES)
    results.append((cfg["name"], m.count_params(), float(tr), float(va), pred_counts.tolist(), cfg))
    print(cfg["name"], "params=", m.count_params(), "train=", round(float(tr),4), "val=", round(float(va),4), "pred_counts=", pred_counts.tolist())

best = sorted(results, key=lambda x: x[3], reverse=True)[0]
print("BEST:", best[0], "val=", round(best[3],4), "params=", best[1])
best_cfg = best[5]

A params= 54010 train= 0.2756 val= 0.2825 pred_counts= [24, 0, 73, 87, 33, 70, 31, 0, 0, 82]
B params= 108154 train= 0.345 val= 0.39 pred_counts= [11, 17, 64, 68, 5, 62, 48, 2, 99, 24]
C params= 70378 train= 0.4194 val= 0.395 pred_counts= [69, 65, 0, 37, 26, 0, 28, 64, 50, 61]
BEST: C val= 0.395 params= 70378
